<h1 align="center">Lab 2:  Sexism Identification in Twitter</h1>
<h2 align="center">Session 1. Machine Learning and Feature Engineering</h2>

<h3 style="display:block; margin-top:5px;" align="center">Natural Language and Information Retrieval</h3>
<h3 style="display:block; margin-top:5px;" align="center">Degree in Data Science</h3>
<h3 style="display:block; margin-top:5px;" align="center">2024-2025</h3>    
<h3 style="display:block; margin-top:5px;" align="center">ETSInf. Universitat Politècnica de València</h3>
<br>

### Put your names here

- Bartosz Stoklosa
- Baranyi Sándor

In [49]:
# Reading the entire dataset for both languages and considering only the hard labels. In this lab we do not address the sexism identification task from a Learning with Disagreement (LwD) perspective.

from readerEXIST2025 import EXISTReader

reader_train = EXISTReader("EXIST_2025_Dataset_V0.2/EXIST2025_training.json")
reader_dev = EXISTReader("EXIST_2025_Dataset_V0.2/EXIST2025_dev.json")

EnTrainTask1, EnDevTask1 = reader_train.get(lang="EN", subtask="1"), reader_dev.get(lang="EN", subtask="1")
EnTrainTask2, EnDevTask2 = reader_train.get(lang="EN", subtask="2"), reader_dev.get(lang="EN", subtask="2")

SpTrainTask1, SpDevTask1 = reader_train.get(lang="ES", subtask="1"), reader_dev.get(lang="ES", subtask="1")
SpTrainTask2, SpDevTask2 = reader_train.get(lang="ES", subtask="2"), reader_dev.get(lang="ES", subtask="2")


# ENGLISH

## Preprocessing

In [50]:
!pip install emoji

In [54]:
def preprocess(text):
    import re, string, emoji

    web_re = re.compile(r"https?:\/\/[^\s]+", re.U)
    user_re = re.compile(r"(@\w+\-?(?:\w+)?)", re.U)
    hashtag_re = re.compile(r"(#\w+\-?(?:\w+)?)", re.U)

    # Apply regex replacements directly to the string
    text = re.sub(user_re, "", text)
    text = re.sub(web_re, "", text)
    text = re.sub(hashtag_re, "", text)
    text = emoji.replace_emoji(text, replace="")
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [69]:
def clean_text_list(texts):
    return [str([preprocess(t).strip()]) for t in texts if t and preprocess(t).strip() != ""]

# Apply to all tasks/languages
EnTrainTask1_cleaned = clean_text_list(EnTrainTask1[1])
EnDevTask1_cleaned   = clean_text_list(EnDevTask1[1])
EnTrainTask2_cleaned = clean_text_list(EnTrainTask2[1])
EnDevTask2_cleaned   = clean_text_list(EnDevTask2[1])
SpTrainTask1_cleaned = clean_text_list(SpTrainTask1[1])
SpDevTask1_cleaned   = clean_text_list(SpDevTask1[1])
SpTrainTask2_cleaned = clean_text_list(SpTrainTask2[1])
SpDevTask2_cleaned   = clean_text_list(SpDevTask2[1])

In [70]:
print(SpTrainTask1_cleaned[:5])  # Sample Spanish tweets after preprocessing

["['ignora al otro es un capulloel problema con este youtuber denuncia el acoso cuando no afecta a la gente de izquierdas por ejemplo en su video sobre el gamergate presenta como normal el acoso que reciben fisher anita o zöey cuando hubo hasta amenazas de bomba']", "['si comicsgate se parece en algo a gamergate pues muy bien por el acoso y si se está haciendo un sabotaje porque hay personajes que no os gustan entonces gracias por darme la razón sois unos lloricas ofendidos']", "['lee sobre gamergate y como eso ha cambiado la manera en la cual nos comunicamos en el internet los fanboys de halo están tóxicos pero los fanboys de otras comunidadesjuegos también han querido coger pauta con eso']", "['entonces como así es el mercado lo mejor no es hacer algo para cambiarlo y seguir alimentando el machismo en los consumidores en lugar apoyar a gente como las víctimas del gamergateacerca de lo otro el tenían implica un imperativo entonces no entiendo lo del buscaban']", "['aaah sí andrew dobs

## Tweet representations (Feature extraction)

In [71]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# Convert labels to numerical format
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(EnTrainTask1[2])  # Binary labels

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(EnTrainTask1_cleaned)  # Training text data

In [78]:
# Convert labels to numerical format
label_encoder_task_2 = LabelEncoder()
y_train_task_2 = label_encoder_task_2.fit_transform(EnTrainTask2[2])  # Binary labels

# TF-IDF Vectorization
vectorizer_task_2 = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf_task_2 = vectorizer_task_2.fit_transform(EnTrainTask2_cleaned)  # Training text data

X_dev_tfidf_task_2 = vectorizer_task_2.transform(EnDevTask2_cleaned)
y_dev_task_2 = label_encoder_task_2.transform(EnDevTask2[2])

## Learning Models

In [72]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report

# Train Logistic Regression
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

# Predict on development set
X_dev_tfidf = vectorizer.transform(EnDevTask1_cleaned)
y_dev = label_encoder.transform(EnDevTask1[2])

y_pred = lr.predict(X_dev_tfidf)

f1_positive	= f1_score(y_dev, y_pred, pos_label=1)
print("Subtask 1")
print(f"F1-score (Positive	Class):	{f1_positive}")
print(classification_report(y_dev, y_pred))

Subtask 1
F1-score (Positive	Class):	0.6826347305389222
              precision    recall  f1-score   support

           0       0.74      0.90      0.81       250
           1       0.81      0.59      0.68       194

    accuracy                           0.76       444
   macro avg       0.78      0.74      0.75       444
weighted avg       0.77      0.76      0.75       444



In [80]:
# Train Logistic Regression
lr2 = LogisticRegression()
lr2.fit(X_train_tfidf_task_2, y_train_task_2)

# Predict on development set
X_dev_tfidf_task_2 = vectorizer_task_2.transform(EnDevTask2_cleaned)
y_dev_task_2 = label_encoder_task_2.transform(EnDevTask2[2])

y_pred_task_2 = lr2.predict(X_dev_tfidf_task_2)

f1_positive	= f1_score(y_dev_task_2, y_pred_task_2, average = 'macro')
print("Subtask 2")
print(f"F1-score (Positive	Class):	{f1_positive}")
print(classification_report(y_dev_task_2, y_pred_task_2))

Subtask 2
F1-score (Positive	Class):	0.32091290172333486
              precision    recall  f1-score   support

           0       0.60      1.00      0.75        85
           1       0.00      0.00      0.00        28
           2       0.80      0.12      0.21        33

    accuracy                           0.61       146
   macro avg       0.47      0.37      0.32       146
weighted avg       0.53      0.61      0.49       146



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [81]:
from sklearn import svm
from sklearn.metrics import f1_score, classification_report
###########################SUBTASK	1##################################
#Using	the	SVC	with	default	settings
clf_en	= svm.SVC()
clf_en.fit(X_train_tfidf, y_train)
predicted = clf_en.predict(X_dev_tfidf)

#Computing	the	F1	Score	for	the	positive	class	(sexism),	subtask	1
print("Subtask 1")
f1_positive	= f1_score(y_dev,	predicted,	pos_label=1)
print(f"F1-score (Positive	Class):	{f1_positive}")

#Obtainig a detailed classification	report
report	= classification_report(y_dev, predicted, digits=4)
print(report)

Subtask 1
F1-score (Positive	Class):	0.7005988023952096
              precision    recall  f1-score   support

           0     0.7467    0.9080    0.8195       250
           1     0.8357    0.6031    0.7006       194

    accuracy                         0.7748       444
   macro avg     0.7912    0.7555    0.7600       444
weighted avg     0.7856    0.7748    0.7675       444



In [82]:
###########################SUBTASK	2##################################
clf_task_2_en	= svm.SVC()
clf_task_2_en.fit(X_train_tfidf_task_2, y_train_task_2)
predicted_task_2 = clf_task_2_en.predict(X_dev_tfidf_task_2)

#Computing	the	macro-average F1 for the second	subtask
f1_macro = f1_score(y_dev_task_2, predicted_task_2,	average='macro')
print("Subtask 2")
print(f"F1-score (Macro-Averaged): {f1_macro}")

#Obtainig a detailed classification	report
report = classification_report(y_dev_task_2, predicted_task_2, digits=4)
print(report)

Subtask 2
F1-score (Macro-Averaged): 0.24531024531024528
              precision    recall  f1-score   support

           0     0.5822    1.0000    0.7359        85
           1     0.0000    0.0000    0.0000        28
           2     0.0000    0.0000    0.0000        33

    accuracy                         0.5822       146
   macro avg     0.1941    0.3333    0.2453       146
weighted avg     0.3389    0.5822    0.4285       146



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [83]:
from sklearn.neural_network	import MLPClassifier
from sklearn.metrics	import f1_score,	classification_report

###########################SUBTASK	1##################################
#Using	the	MLP	with	default	settings
clf_en	=	MLPClassifier(random_state=1,	max_iter=300)
clf_en.fit(X_train_tfidf,	y_train)
predicted1	=	clf_en.predict(X_dev_tfidf)

#Computing	the	F1	Score	for	the	positive	class	(sexism),	subtask	1
f1_positive	=	f1_score(y_dev,	predicted1,	pos_label=1)
print("Subtask 1")
print(f"F1-score	(Positive	Class):	{f1_positive}")

#Obtainig	a	detailed	classification	report
report	=	classification_report(y_dev, predicted1,	digits=4)
print(report)

###########################SUBTASK	2##################################
clf_task_2_en	=	MLPClassifier(random_state=1,	max_iter=300)
clf_task_2_en.fit(X_train_tfidf_task_2,	y_train_task_2)
predicted_task_2	=	clf_task_2_en.predict(X_dev_tfidf_task_2)

#Computing	the	macro-average	F1	for	the	second	subtask
f1_macro	=	f1_score(y_dev_task_2,	predicted_task_2,	average='macro')
print("Subtask 2")
print(f"F1-score	(Macro-Averaged):	{f1_macro}")

#Obtainig	a	detailed	classification	report
report	=	classification_report(y_dev_task_2, predicted_task_2,	digits=4)
print(report)

Subtask 1
F1-score	(Positive	Class):	0.6070460704607046
              precision    recall  f1-score   support

           0     0.6952    0.7480    0.7206       250
           1     0.6400    0.5773    0.6070       194

    accuracy                         0.6734       444
   macro avg     0.6676    0.6627    0.6638       444
weighted avg     0.6711    0.6734    0.6710       444

Subtask 2
F1-score	(Macro-Averaged):	0.45408163265306123
              precision    recall  f1-score   support

           0     0.6667    0.8706    0.7551        85
           1     0.4167    0.1786    0.2500        28
           2     0.4348    0.3030    0.3571        33

    accuracy                         0.6096       146
   macro avg     0.5060    0.4507    0.4541       146
weighted avg     0.5663    0.6096    0.5683       146



# SPANISH

## Preprocessing

In [85]:
!pip install stopwordsiso

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.1 MB/s eta 0:00:00


In [86]:
from sklearn.feature_extraction.text import TfidfVectorizer
from stopwordsiso import stopwords  # Install via: pip install stopwordsiso
spanish_stopwords = stopwords("es")  # Get Spanish stop words

## Tweet representations (Feature extraction)

In [89]:
# Obtaining a representation for the train and dev subsets in both tasks

# Convert labels to numerical format
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(SpTrainTask1[2])  # Binary labels

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words=list(spanish_stopwords),max_features=5000)
X_train_tfidf = vectorizer.fit_transform(SpTrainTask1_cleaned)  # Training text data

X_dev_tfidf = vectorizer.transform(SpDevTask1_cleaned)
y_dev = label_encoder.transform(SpDevTask1[2])

# Convert labels to numerical format
label_encoder_task_2 = LabelEncoder()
y_train_task_2 = label_encoder_task_2.fit_transform(SpTrainTask2[2])  # Binary labels

# TF-IDF Vectorization
vectorizer_task_2 = TfidfVectorizer(stop_words=list(spanish_stopwords), max_features=5000)
X_train_tfidf_task_2 = vectorizer_task_2.fit_transform(SpTrainTask2_cleaned)  # Training text data

X_dev_tfidf_task_2 = vectorizer_task_2.transform(SpDevTask2_cleaned)
y_dev_task_2 = label_encoder_task_2.transform(SpDevTask2[2])

## Learning Models

In [91]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred = lr.predict(X_dev_tfidf)

f1_positive	= f1_score(y_dev, y_pred, pos_label=1)
print("Subtask 1")
print(f"F1-score (Positive	Class):	{f1_positive}")
print(classification_report(y_dev, y_pred))

Subtask 1
F1-score (Positive	Class):	0.7644628099173554
              precision    recall  f1-score   support

           0       0.72      0.83      0.77       229
           1       0.83      0.71      0.76       261

    accuracy                           0.77       490
   macro avg       0.77      0.77      0.77       490
weighted avg       0.78      0.77      0.77       490



In [92]:
lr_task_2 = LogisticRegression()
lr_task_2.fit(X_train_tfidf_task_2, y_train_task_2)

y_pred_task_2 = lr_task_2.predict(X_dev_tfidf_task_2)

f1_positive	= f1_score(y_dev_task_2, y_pred_task_2, average='macro')
print("Subtask 2")
print(f"F1-score (Positive	Class):	{f1_positive}")
print(classification_report(y_dev_task_2, y_pred_task_2))

Subtask 2
F1-score (Positive	Class):	0.34456992283079235
              precision    recall  f1-score   support

           0       0.60      0.99      0.75       116
           1       0.67      0.12      0.20        51
           2       0.33      0.05      0.09        40

    accuracy                           0.59       207
   macro avg       0.53      0.39      0.34       207
weighted avg       0.56      0.59      0.48       207



In [93]:
###########################SUBTASK	1##################################
#Using	the	SVC	with	default	settings
clf1_es	= svm.SVC()
clf1_es.fit(X_train_tfidf, y_train)
predicted = clf1_es.predict(X_dev_tfidf)

#Computing	the	F1	Score	for	the	positive	class	(sexism),	subtask	1
print("Subtask 1")
f1_positive	= f1_score(y_dev,	predicted,	pos_label=1)
print(f"F1-score (Positive	Class):	{f1_positive}")

#Obtainig a detailed classification	report
report	= classification_report(y_dev, predicted, digits=4)
print(report)

Subtask 1
F1-score (Positive	Class):	0.7541666666666667
              precision    recall  f1-score   support

           0     0.7048    0.8341    0.7640       229
           1     0.8265    0.6935    0.7542       261

    accuracy                         0.7592       490
   macro avg     0.7656    0.7638    0.7591       490
weighted avg     0.7696    0.7592    0.7588       490



In [94]:
clf_task_2_es	= svm.SVC()
clf_task_2_es.fit(X_train_tfidf_task_2, y_train_task_2)
predicted = clf_task_2_es.predict(X_dev_tfidf_task_2)

#Computing	the	macro-average F1 for the second	subtask
f1_macro = f1_score(y_dev_task_2, predicted,	average='macro')
print("Subtask 2")
print(f"F1-score (Macro-Averaged): {f1_macro}")

#Obtainig a detailed classification	report
report = classification_report(y_dev_task_2, predicted, digits=4)
print(report)

Subtask 2
F1-score (Macro-Averaged): 0.2960966357192772
              precision    recall  f1-score   support

           0     0.5743    1.0000    0.7296       116
           1     1.0000    0.0588    0.1111        51
           2     0.5000    0.0250    0.0476        40

    accuracy                         0.5797       207
   macro avg     0.6914    0.3613    0.2961       207
weighted avg     0.6648    0.5797    0.4454       207



In [95]:
from sklearn.neural_network	import MLPClassifier
from sklearn.metrics	import f1_score,	classification_report

###########################SUBTASK	1##################################
#Using	the	MLP	with	default	settings
clf_es	=	MLPClassifier(random_state=1,	max_iter=300)
clf_es.fit(X_train_tfidf,	y_train)
predicted	=	clf_es.predict(X_dev_tfidf)

#Computing	the	F1	Score	for	the	positive	class	(sexism),	subtask	1
f1_positive	=	f1_score(y_dev,	predicted,	pos_label=1)
print("Subtask 1")
print(f"F1-score	(Positive	Class):	{f1_positive}")

#Obtainig	a	detailed	classification	report
report	=	classification_report(y_dev, predicted,	digits=4)
print(report)

###########################SUBTASK	2##################################
clf_task_2_es	=	MLPClassifier(random_state=1,	max_iter=300)
clf_task_2_es.fit(X_train_tfidf_task_2,	y_train_task_2)
predicted	=	clf_task_2_es.predict(X_dev_tfidf_task_2)

#Computing	the	macro-average	F1	for	the	second	subtask
f1_macro	=	f1_score(y_dev_task_2,	predicted,	average='macro')
print("Subtask 2")
print(f"F1-score	(Macro-Averaged):	{f1_macro}")

#Obtainig	a	detailed	classification	report
report	=	classification_report(y_dev_task_2, predicted,	digits=4)
print(report)

Subtask 1
F1-score	(Positive	Class):	0.7065868263473054
              precision    recall  f1-score   support

           0     0.6640    0.7249    0.6931       229
           1     0.7375    0.6782    0.7066       261

    accuracy                         0.7000       490
   macro avg     0.7007    0.7015    0.6998       490
weighted avg     0.7031    0.7000    0.7003       490

Subtask 2
F1-score	(Macro-Averaged):	0.4660379615007564
              precision    recall  f1-score   support

           0     0.6690    0.8362    0.7433       116
           1     0.3462    0.1765    0.2338        51
           2     0.4444    0.4000    0.4211        40

    accuracy                         0.5894       207
   macro avg     0.4865    0.4709    0.4660       207
weighted avg     0.5460    0.5894    0.5555       207



**Preprocessing**

For all the tweets in English and Spanish (Task 1 and 2), we applied the following steps to clean the text:
- Removed @mentions
- Removed URLs
- Removed hashtags
- Removed emojis
- Converted text to lowercase
- Removed punctuation
- Removed extra spaces

This was done using a custom preprocess() function before converting the text into features.

**Text Representation**

We used TF-IDF (Term Frequency-Inverse Document Frequency) to convert the cleaned text into numeric features.
The vectorizer was set to:
- stop_words='english'
- max_features=5000

**Classification**

We trained Logistic Regression models using scikit-learn.

Task 1:
- Binary classification (YES vs NO)
- Labels were encoded with LabelEncoder

Task 2:
- Multi-class classification (JUDGEMENTAL, REPORTED, DIRECT)
- Same process as Task 1

**Hyperparameters**

We didn’t do any tuning. All models used default settings.
- TF-IDF: stop_words='english', max_features=5000
- Classifier: LogisticRegression() with default config

**Results**

Task 1 (English):
- Evaluated using F1-score on the positive class (YES)
- We printed the full classification report (precision, recall, F1, support)

Task 2 (English):
- Same setup, but with 3-class output

Note: We preprocessed the Spanish tasks too, but didn’t include evaluation for them yet.
